In [2]:
from pathlib import Path
from typing import Literal
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

from python.error_metrics import MergedErrorMetrics, LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS

In [6]:
DATASET: Literal['web-traffic', 'M5', 'M4'] = 'M5'
MODEL: Literal['AutoETS', 'AutoARIMA', 'AutoTheta', 'Prophet'] = 'AutoARIMA'

FOLDER = f"{DATASET}/{MODEL}"

Path(f"metrics/{FOLDER}").mkdir(parents=True, exist_ok=True)

In [7]:
from pathlib import Path
ems = []
for file in sorted(Path(f"metrics/{FOLDER}").rglob("sample_hiers_errors_raw*.csv")):
    if file.is_file():
        ems.append(LoadedErrorMetrics(file))

em = MergedErrorMetrics(ems, 'horizontal')

em.save(f'metrics/{FOLDER}/combined_sample_hiers_errors_raw.csv')

In [8]:
em = LoadedErrorMetrics(f'metrics/{FOLDER}/combined_sample_hiers_errors_raw.csv')

em = MergedErrorMetrics([
    FilteredErrorMetrics(em, MI(forecast_type=RECONCILED_FORECASTS)),
    FilteredErrorMetrics(em, MI(forecast_type=BASE_FORECASTS, hierarchy='Minimal'))
])

em.save(f'metrics/{FOLDER}/sample_hiers_errors.csv')